Libraries

In [1]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report

Loading the dataset

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/mohan13krishna/Student-Academic-Performance-Dropout-Risk/main/student_dropout_risk_2026.csv")
df.head()

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,...,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,...,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,...,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,4,7.4,1,0,7,0,10,24,5.8,0


Data exploration

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 34 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   person_id                     100000 non-null  int64  
 1   age                           100000 non-null  int64  
 2   gender                        100000 non-null  object 
 3   nationality                   100000 non-null  object 
 4   first_generation              100000 non-null  int64  
 5   disability_status             100000 non-null  int64  
 6   family_income_level           100000 non-null  object 
 7   scholarship_holder            100000 non-null  int64  
 8   tuition_up_to_date            100000 non-null  int64  
 9   part_time_job                 100000 non-null  int64  
 10  weekly_work_hours             100000 non-null  int64  
 11  course_type                   100000 non-null  object 
 12  study_mode                    100000 non-null

In [4]:
df.nunique()

person_id                       100000
age                                 16
gender                               3
nationality                         15
first_generation                     2
disability_status                    2
family_income_level                  3
scholarship_holder                   2
tuition_up_to_date                   2
part_time_job                        2
weekly_work_hours                   37
course_type                          5
study_mode                           3
semester                             8
international_student                2
admission_grade                   1602
previous_qualification_grade      1449
units_enrolled                      24
units_passed                        24
units_failed                        18
gpa_semester_1                     400
gpa_semester_2                     401
attendance_rate                    889
assignments_submitted_rate         977
library_visits_per_month            31
lms_logins_per_week      

In [5]:
non_numeric_columns = df.select_dtypes(exclude="number").columns
print(non_numeric_columns.tolist())

['gender', 'nationality', 'family_income_level', 'course_type', 'study_mode']


non-numerical values = gender - nationality - family_income_level - course_type - study_mode                           

In [6]:
categorical_features = [
    "gender",
    "nationality",
    "family_income_level",
    "course_type",
    "study_mode"
]

Preprocessing

In [7]:
X = df.drop(columns = "dropout")
y = df["dropout"]
X

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,assignments_submitted_rate,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,75.0,4,16.3,0,3,8,0,6,37,5.7
1,2,18,Other,USA,1,1,High,1,1,0,...,80.1,3,8.4,1,1,7,0,6,15,5.4
2,3,24,Other,USA,0,0,High,0,1,1,...,67.8,3,11.3,1,0,2,0,3,4,4.3
3,4,25,Female,USA,0,1,Middle,1,1,0,...,100.0,2,13.2,1,1,5,0,7,10,6.7
4,5,15,Female,Canada,0,0,Low,1,1,0,...,100.0,4,7.4,1,0,7,0,10,24,5.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,25,Female,USA,0,0,Middle,1,1,1,...,100.0,10,6.2,0,0,4,0,6,11,5.9
99996,99997,24,Female,India,0,1,Middle,0,1,1,...,32.9,0,8.7,1,3,7,1,5,21,5.5
99997,99998,27,Female,USA,0,0,Middle,1,1,1,...,88.9,8,7.5,0,2,2,0,6,29,5.8
99998,99999,21,Male,UK,0,0,High,0,1,0,...,62.4,6,13.6,1,2,4,1,10,43,6.2


Train Test Split

In [8]:
train_X, test_X, train_y, test_y = train_test_split(
    X,
    y,
    test_size=0.3, 
    stratify=y, 
    random_state=42
)

Baseline model: SGD classifier with 5 folds

In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

baseline_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(train_X, train_y), start=1):
    fold_train_X = train_X.iloc[train_idx]
    fold_val_X = train_X.iloc[val_idx]

    fold_train_y = train_y.iloc[train_idx]
    fold_val_y = train_y.iloc[val_idx]

    transformer = ColumnTransformer(
        transformers=[
            (
                "one_hot",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_features
            )
        ],
        remainder="passthrough"
    )

    fold_train_encoded_X = transformer.fit_transform(fold_train_X)
    fold_val_encoded_X = transformer.transform(fold_val_X)

    scaler = StandardScaler(with_mean=False)

    fold_train_scaled_X = scaler.fit_transform(fold_train_encoded_X)
    fold_val_scaled_X = scaler.transform(fold_val_encoded_X)

    clf_baseline = SGDClassifier(
        loss="hinge",
        random_state=42
    )

    clf_baseline.fit(fold_train_scaled_X, fold_train_y)

    val_predictions = clf_baseline.predict(fold_val_scaled_X)

    score = accuracy_score(fold_val_y, val_predictions)
    baseline_scores.append(score)

    print(f"Fold {fold} validation accuracy: {score:.2%}")


Fold 1 validation accuracy: 81.64%
Fold 2 validation accuracy: 83.61%
Fold 3 validation accuracy: 84.02%
Fold 4 validation accuracy: 83.50%
Fold 5 validation accuracy: 83.01%


In [10]:
baseline_transformer = ColumnTransformer(
    transformers=[
        (
            "one_hot",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

train_encoded_X = baseline_transformer.fit_transform(train_X)
test_encoded_X = baseline_transformer.transform(test_X)

In [11]:
baseline_scaler = StandardScaler(with_mean=False)

train_scaled_encoded_X = baseline_scaler.fit_transform(train_encoded_X)
test_scaled_encoded_X = baseline_scaler.transform(test_encoded_X)

In [12]:
baseline_model = SGDClassifier(
    loss="hinge",
    random_state=42
)

baseline_model.fit(train_scaled_encoded_X, train_y)

test_predictions = baseline_model.predict(test_scaled_encoded_X)

baseline_test_accuracy = accuracy_score(test_y, test_predictions)

print(f"Mean baseline CV validation accuracy: {np.mean(baseline_scores):.2%}")
print(f"Baseline test accuracy: {baseline_test_accuracy:.2%}")
print(classification_report(test_y, test_predictions))

Mean baseline CV validation accuracy: 83.15%
Baseline test accuracy: 83.84%
              precision    recall  f1-score   support

           0       0.87      0.90      0.88     20399
           1       0.77      0.70      0.74      9601

    accuracy                           0.84     30000
   macro avg       0.82      0.80      0.81     30000
weighted avg       0.84      0.84      0.84     30000

